In [1]:
import pandas as pd 
import numpy as np

In [2]:
data = pd.read_csv('train.csv')

In [3]:
m,n=data.shape

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42000 entries, 0 to 41999
Columns: 785 entries, label to pixel783
dtypes: int64(785)
memory usage: 251.5 MB


In [5]:
data=np.array(data)

In [6]:
import numpy as np

def data_split(data, train_size=0.8, dev_size=0.1):
    data = data.copy()
    np.random.shuffle(data)

    n = len(data)
    train_end = int(train_size * n)
    dev_end = train_end + int(dev_size * n)

    train_data = data[:train_end]
    dev_data = data[train_end:dev_end]
    test_data = data[dev_end:]

    return train_data, dev_data, test_data

In [7]:
train_data, dev_data, test_data = data_split(data)

# العمود الأول هو label، وباقي الأعمدة هي قيم pixels
X_train = train_data[:, 1:].T.astype(float) / 255.0
Y_train = train_data[:, 0]

X_dev = dev_data[:, 1:].T.astype(float) / 255.0
Y_dev = dev_data[:, 0]

X_test = test_data[:, 1:].T.astype(float) / 255.0
Y_test = test_data[:, 0]

print("Train:", X_train.shape, Y_train.shape)
print("Dev:", X_dev.shape, Y_dev.shape)
print("Test:", X_test.shape, Y_test.shape)

Train: (784, 33600) (33600,)
Dev: (784, 4200) (4200,)
Test: (784, 4200) (4200,)


In [8]:
def he_uniform(n_in, n_out, rng):
    limit = np.sqrt(6.0 / n_in)
    return rng.uniform(-limit, limit, size=(n_out, n_in))

In [9]:
def one_hot(Y, num_classes=None):
    Y = np.asarray(Y, dtype=int).reshape(-1)
    if num_classes is None:
        num_classes = int(Y.max()) + 1
    hoted = np.zeros((num_classes, Y.size))
    hoted[Y, np.arange(Y.size)] = 1
    return hoted

In [10]:
def init_params(input_size, hidden_size, output_size, seed=None):
    rng = np.random.default_rng(seed)
    W1 = he_uniform(input_size, hidden_size, rng)
    b1 = np.zeros((hidden_size, 1))
    W2 = he_uniform(hidden_size, output_size, rng)
    b2 = np.zeros((output_size, 1))
    return W1, b1, W2, b2

In [11]:
def relu(z):
    return np.maximum(0, z)

In [12]:
def relu_derivative(z):
    return (z > 0).astype(float)

In [13]:
def softmax(z):
    exp_z=(np.exp(z-np.max(z,axis=0,keepdims=True)))
    return exp_z / np.sum(exp_z,axis=0,keepdims=True)

In [14]:
def forward_prop(W1, b1, W2, b2, X):
    Z1 = W1 @ X + b1
    A1 = relu(Z1)
    Z2 = W2 @ A1 + b2
    A2 = softmax(Z2)
    return Z1, A1, Z2, A2

In [15]:
def backward_prop(Z1, A1, Z2, A2, W1, W2, X, Y):
    one_hot_Y = one_hot(Y, A2.shape[0])
    sample_count = X.shape[1]
    dZ2 = A2 - one_hot_Y
    dW2 = (dZ2 @ A1.T) / sample_count
    db2 = np.sum(dZ2, axis=1, keepdims=True) / sample_count
    dZ1 = (W2.T @ dZ2) * relu_derivative(Z1)
    dW1 = (dZ1 @ X.T) / sample_count
    db1 = np.sum(dZ1, axis=1, keepdims=True) / sample_count
    return dW1, db1, dW2, db2

In [16]:
def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 = W1 - alpha * dW1
    b1 = b1 - alpha * db1    
    W2 = W2 - alpha * dW2  
    b2 = b2 - alpha * db2    
    return W1, b1, W2, b2

In [17]:
import numpy as np

In [18]:
def get_predictions(A2):
    return np.argmax(A2, 0)

def get_accuracy(predictions, Y):
    predictions = np.asarray(predictions).reshape(-1)
    Y = np.asarray(Y).reshape(-1)
    return np.mean(predictions == Y)

In [19]:
def gradient_descent(X, Y, alpha, iterations, hidden_size=10, batch_size=128, seed=0):
    Y = np.asarray(Y, dtype=int).reshape(-1)
    sample_count = X.shape[1]
    output_size = int(Y.max()) + 1
    W1, b1, W2, b2 = init_params(X.shape[0], hidden_size, output_size, seed)
    rng = np.random.default_rng(seed)

    for i in range(iterations):
        batch_indices = rng.choice(sample_count, size=min(batch_size, sample_count), replace=False)
        X_batch = X[:, batch_indices]
        Y_batch = Y[batch_indices]

        Z1, A1, Z2, A2 = forward_prop(W1, b1, W2, b2, X_batch)
        dW1, db1, dW2, db2 = backward_prop(Z1, A1, Z2, A2, W1, W2, X_batch, Y_batch)
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha)

        if i % 50 == 0:
            predictions = get_predictions(A2)
            print(f"Iteration {i}: batch accuracy = {get_accuracy(predictions, Y_batch):.3f}")

    return W1, b1, W2, b2

In [20]:
W1, b1, W2, b2 = gradient_descent(X_train, Y_train, alpha=0.10, iterations=500)

Iteration 0: batch accuracy = 0.172
Iteration 50: batch accuracy = 0.719
Iteration 100: batch accuracy = 0.875


Iteration 150: batch accuracy = 0.891
Iteration 200: batch accuracy = 0.797
Iteration 250: batch accuracy = 0.875
Iteration 300: batch accuracy = 0.898
Iteration 350: batch accuracy = 0.844
Iteration 400: batch accuracy = 0.883
Iteration 450: batch accuracy = 0.922


In [21]:
_, _, _, A2_dev = forward_prop(W1, b1, W2, b2, X_dev)
dev_predictions = get_predictions(A2_dev)
print("Dev accuracy:", get_accuracy(dev_predictions, Y_dev))

_, _, _, A2_test = forward_prop(W1, b1, W2, b2, X_test)
test_predictions = get_predictions(A2_test)
print("Test accuracy:", get_accuracy(test_predictions, Y_test))

Dev accuracy: 0.8866666666666667
Test accuracy: 0.8752380952380953


In [22]:
def accuracy(y_true, y_pred):
    correct = sum(1 for a, b in zip(y_true, y_pred) if a == b)
    total = len(y_true)
    return correct / total if total else 0